# Ear-Voice Span in Simultaneous Interpreting with Text

## Module 04: Lesson 01-01

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Critt-Kent/CRITT-Academy/blob/main/modules/04_Exploring_Modalities/01_01_Ear_Voice_Span.ipynb) | [This video linked here](https://vimeo.com/1201603642) explains how to use this code notebook in Google Colab 🤠

### What You Will Do In This Lesson

1. Install and import all necessary Python libraries
2. Load source text (ST) data tables from the TPR-DB for the **IMB18** study (simultaneous interpreting with text)
3. Read them into a `pandas` DataFrame
4. Inspect the data and clean the DataFrame
5. Compute temporal spans: Ear-Eye Span (EIS), Eye-Voice Span (IVS), and Ear-Voice Span (EVS)
6. Visualize the span distributions and explore the EVS–IVS correlation

### First time working with a CRITT Academy code notebook?

If you haven’t yet gone through the [CRITT Academy Module 01, Lesson 01, “Code Notebooks and the TPR-DB”](https://github.com/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/01_Notebooks_and_TPRDB.ipynb), you should do that first (this will help you understand Steps 1 through 3 much better).

## Step 1. Install and import all necessary Python libraries

**Python libraries** are just packages of code that are used for a specific purpose. Here are the Python libraries that are used by a typical CRITT Academy lesson:

- **NumPy**: `numpy` is used for scientific computing and data science
- **Pandas**: `pandas` is used for data manipulation, organization, and analysis
- **SciPy `stats`**: the `stats` module within `scipy` is used for statistical analysis and probability distributions
- **Matplotlib**: `matplotlib` is used for data visualization
- **Seaborn**: `seaborn` is used for nice-looking statistical graphics
- **tprdb-utilities**: `tprdb_utilities` is used for getting data remotely from the CRITT TPR-DB and then for reading data tables from a certain study (or studies) into a Pandas DataFrame.

### So what does the following code in Step 1 do?

The code in this step is something you will see in all CRITT Academy lessons.

1. First, you make sure you have all the Python libraries installed that will be necessary for the lesson.
2. Then, you *import* the Python libraries so they are ready to use for the entire rest of the notebook

In [ ]:
# Environment Setup
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

# Enforce a minimum Python version of 3.9+
if sys.version_info < (3, 9):
    raise RuntimeError(
        f"❌ Python 3.9 or higher is required. "
        f"You are currently running Python {sys.version_info.major}.{sys.version_info.minor}."
    )

REQUIRED = {
    "numpy": "2.4.0",
    "pandas": "3.0.0",
    "scipy": "1.17.0",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.0",
    "tprdb-utilities": "0.3.0",
}

def _check_versions():
    outdated = []
    for pkg, min_ver in REQUIRED.items():
        try:
            installed = version(pkg)
            if Version(installed) < Version(min_ver):
                outdated.append(f"  {pkg}: installed {installed}, need >={min_ver}")
        except PackageNotFoundError:
            raise ImportError(f"{pkg} is not installed")
    return outdated

# Check if dependencies are missing and install them automatically
try:
    import numpy as np
    import pandas as pd
    from scipy import stats
    import matplotlib.pyplot as plt
    from matplotlib import rcParams
    import seaborn as sns
    import tprdb_utilities

    outdated = _check_versions()
    if outdated:
        raise ImportError("Outdated packages:\n" + "\n".join(outdated))
    else:
        print("🤠 All core dependencies are already installed. You are ready to go! 🤘")
except ImportError:
    print("Missing dependencies detected. Installing required packages...")

    try:
        # The %pip magic ensures installation happens in the active Jupyter kernel
        %pip install "numpy>=2.4.0" "pandas>=3.0.0" "scipy>=1.17.0" "matplotlib>=3.10.0" "seaborn>=0.13.0" "tprdb-utilities>=0.3.0"

        print("\n🤠 Installation complete 🤘 If imports fail on the next cell, please restart the kernel.")
    except Exception as e:
        print(f"❌ An error occurred during installation: {e}")
        print("If using Google Colab, you may just have to restart your runtime now to use the newly installed packages.")

In [ ]:
# Now, import the libraries

try:
    # Standard Python library imports
    import numpy as np
    import pandas as pd
    from scipy import stats
    import matplotlib.pyplot as plt
    from matplotlib import rcParams
    import seaborn as sns

    # TPR-DB utilities import
    from tprdb_utilities import fetch_TPRDB_tables, read_TPRDB_tables

    print("✅ All imports successful!")

except ImportError as e:
    print(f"❌ An error occurred during imports: {e}")
    print("Please ensure all dependencies are installed and the kernel is restarted if necessary.")

## Step 2: Load Source Text (ST) Data Tables

In this lesson, we work with the **IMB18** study from the TPR-DB, which contains data from a **simultaneous interpreting with text** experiment. This study combines both ear (audio) and eye (gaze) source text data:

- When each ST word was **spoken** to the interpreter (ear/audio data)
- When each ST word was **fixated** by the interpreter (eye/gaze data)
- When each TT word was **spoken** by the interpreter (voice/production data)

### 🫵 Change code cell below

To get ready to run the `fetch_TPRDB_tables()` function, change the code cell below depending on your situation:
- **If *NOT* using Google Colab**:
    - Follow the **1st** set of directions to change the value of the `mothership_clone_location` variable
- **If you *are* using Google Colab**:
    - Follow the **2nd** set of directions to
        1. Comment out the code from the 1st set of directions
        2. Uncomment the code from the 2nd set of directions
        3. mount your Google Drive (unless you want it in a more specific location, you shouldn’t need to change the value of the `mothership_clone_location` variable after uncommenting the code)

[Link to 📽️ Video on How to Comment Out and Uncomment Code](https://vimeo.com/1201588460)

#### ⚠️ If you are unsure what is going on here..

If you are confused about this step, it means you should go back and review [CRITT Academy Module 01, Lesson 01, “Code Notebooks and the TPR-DB”](https://github.com/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/01_Notebooks_and_TPRDB.ipynb).

In [ ]:
# Commenting/Uncommenting Code: ⌘ + / (Mac) or Ctrl + / (Windows/Linux)

# 🫵 If NOT using Google Colab:
# 1) Change what's in the quotes below ("./") to point to where you want your mothership clone to be saved.
# 2) Then, run this code cell
mothership_clone_location = "./"

# 🫵 If using Google Colab:
# 1) Comment out the above line of code
# 2) Uncomment the lines of code below
# 3) Then, run this code cell

# from google.colab import drive
# drive.mount('/content/drive')
# mothership_clone_location = "/content/drive/MyDrive/"

In [ ]:
# fetch ST data tables for the IMB18 study

fetch_TPRDB_tables(
    path=mothership_clone_location,
    studies=["IMB18"],
    extensions=["st"],
    public=True,
)

## Step 3: Read the ST table into a DataFrame

You will now read the ST table into a DataFrame using the `read_TPRDB_tables()` function. You will assign the output of the function to a variable called `st`, so `st` is now the name of the DataFrame containing the ST table.

In [ ]:
# read the ST table into a DataFrame

st = read_TPRDB_tables(
    studies=["IMB18"],
    extension="st",
    path=f"{mothership_clone_location}/tprdb-mothership-clone",
    user="PUBLIC",
    verbose=1,
)

In [ ]:
# Check the shape of the DataFrame and list all columns

print(f"📊 The ST table has {st.shape[0]} rows and {st.shape[1]} columns")
print(f"\nColumns: {list(st.columns)}")

## Step 4: Inspect the data

Let’s look at the first few rows to understand the structure of the data and identify the key timing columns we need for computing spans.

In [ ]:
# Inspect the first few rows

with pd.option_context("display.max_columns", None):
    display(st.head())

### Key timing columns

For computing ear-voice and eye-voice spans, we need three timing columns:

- **STime**: time when the ST word was **spoken** (heard by the interpreter)
- **FFTimeS**: time when the ST word was **first fixated** (looked at by the interpreter)
- **Time1**: time when the TT word was **spoken** (produced by the interpreter)

## Step 5: Clean the data

Before computing spans, we need to remove rows that would produce meaningless results. We filter on exactly the three columns used in the span computation:

- There is **no translation** (the `TGroup` column is empty / `---`)
- There is **no fixation** (`FFTimeS` is zero)

We do not need to filter on `STime` or `Time1` because both are already non-zero for all translated rows.

In [ ]:
# Take out words with no translation (TGroup empty)

shape = st.shape
st1 = st[st['TGroup'] != '---']

print(f"rows before: {shape[0]}  after: {st1.shape[0]}  deleted: {shape[0] - st1.shape[0]}")

In [ ]:
# Take out words with no fixation (FFTimeS == 0)

shape = st1.shape
st1 = st1[st1['FFTimeS'] > 0]

print(f"rows before: {shape[0]}  after: {st1.shape[0]}  deleted: {shape[0] - st1.shape[0]}")

## Step 6: Compute temporal spans

Now we compute three important temporal spans that characterize the interpreter’s processing:

- **Ear-Eye Span (EIS)**: `STime - FFTimeS` — the time difference between when a word was *heard* and when it was *fixated*
- **Eye-Voice Span (IVS)**: `Time1 - FFTimeS` — the time difference between when a word was *fixated* and when its translation was *spoken*
- **Ear-Voice Span (EVS)**: `Time1 - STime` — the time difference between when a word was *heard* and when its translation was *spoken*

In [ ]:
# Compute the three spans

st1['EIS'] = st1['STime'] - st1['FFTimeS']
st1['IVS'] = st1['Time1'] - st1['FFTimeS']
st1['EVS'] = st1['Time1'] - st1['STime']

# Show descriptive statistics for the three spans
st1[['EIS', 'IVS', 'EVS']].describe()

## Step 7: Visualize the span distributions

Let’s look at the distributions of the **Eye-Voice Span (IVS)** and **Ear-Eye Span (EIS)** using histograms.

In [ ]:
# Histogram of Eye-Voice Span (IVS)

sns.set(font_scale=2)
sns.set_style("white")
rcParams['figure.figsize'] = 12, 8

data = st1['IVS']
mean = np.mean(data)

plt.hist(data, bins=50, color='#0504aa', alpha=0.7, rwidth=0.85)
plt.grid(axis='y', alpha=0.75)
plt.xlabel('Eye-Voice Span (IVS)')
plt.ylabel('Frequency')
plt.text(25000, 500, f'$\mu$={mean:.2f}')
plt.show()

In [ ]:
# Histogram of Ear-Eye Span (EIS)

data = st1['EIS']
mean = np.mean(data)

plt.hist(data, bins=50, color='#0504aa', alpha=0.7, rwidth=0.85)
plt.grid(axis='y', alpha=0.75)
plt.xlabel('Ear-Eye Span (EIS)')
plt.ylabel('Frequency')
plt.text(25000, 500, f'$\mu$={mean:.2f}')
plt.show()

## Step 8: Explore the EVS–IVS correlation

Finally, let’s look at the relationship between the **Ear-Voice Span (EVS)** and the **Eye-Voice Span (IVS)** using a joint plot with a regression line and Pearson correlation.

In [ ]:
# Joint plot: EVS vs. IVS with Pearson correlation

sns.set(font_scale=2)
sns.set_style("white")

res = stats.pearsonr(x=st1['EVS'], y=st1['IVS'])

g = sns.JointGrid(data=st1, x='EVS', y='IVS', height=12, ratio=3, space=0.1)
g.plot_joint(sns.regplot, color="g")
g.plot_marginals(sns.histplot, kde=True)
g.ax_joint.text(25000, 60000, f'\nr={res[0]:4.4f}\np={res[1]:4.4f}', size=20, color='purple')
plt.show()

---

## Citations

If you use this notebook or the analysis approach it demonstrates, please cite:

> Zou, L., Carl, M., & Feng, J. (2022). Patterns of attention and quality in English-Chinese simultaneous interpreting with text. *International Journal of Chinese and English Translation & Interpreting*.

```bibtex
@article{zou2022patterns,
  title={Patterns of attention and quality in English-Chinese simultaneous interpreting with text},
  author={Zou, Longhui and Carl, Michael and Feng, Jia},
  journal={International Journal of Chinese and English Translation \& Interpreting},
  year={2022}
}
```

For published studies that also used this dataset, or for further descriptions of the dataset, see:

> Zou, L., Carl, M., Mirzapour, M., Jacquenet, H., & Vieira, L. N. (2021). AI-based syntactic complexity metrics and sight interpreting performance. In *International Conference on Intelligent Human Computer Interaction* (pp. 534–547). Springer.

```bibtex
@inproceedings{zou2021ai,
  title={AI-based syntactic complexity metrics and sight interpreting performance},
  author={Zou, Longhui and Carl, Michael and Mirzapour, Mehdi and Jacquenet, H{\'e}l{\`e}ne and Vieira, Lucas Nunes},
  booktitle={International Conference on Intelligent Human Computer Interaction},
  pages={534--547},
  year={2021},
  organization={Springer}
}
```